# 04 · Operator logbook sentiment — per shot, not per session

Parses each session's electronic logbook (`summary.html`). Every entry is tagged `Shot <number> <ROLE>:`, so notes can be attributed to the **individual shot** they describe.

* **Tier 1 — per-shot notes:** scored from that shot's own SESSION_LEADER / PHYSICS_OPERATOR / CHIEF_OPERATOR / DIAGNOSTICS / BEAMS entries.
* **Tier 2 — session fallback:** shots with no individual entry inherit the session-wide narrative sentiment, and are **labeled as such** so they are never treated as equal-quality signal.

Keyword lists were tuned for plasma-physics vocabulary (e.g. *mode* and *clean* removed because of false hits on "O-mode" and "wall cleaning").

**Result:** 3,062 sessions parsed → 53,384 shots scored (40,859 from their own notes, 12,525 via session fallback).

In [ ]:
import os
import re
import csv

print("Imports done")

In [ ]:
full_logbook_path = "/path/to/big_d3d_data/foundation_model_text/shotsummary/raw"
session_folders = sorted(os.listdir(full_logbook_path))
print("Session folders found:", len(session_folders))

In [ ]:
def clean_html(html_text):
    text = re.sub(r'<[^>]+>', ' ', html_text)
    text = re.sub(r'&nbsp;', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text

SHOT_ENTRY_PATTERN = re.compile(r'Shot (\d+) ([A-Z_]+):')

def parse_session_html_by_shot(clean_text, relevant_roles=None):
    """Per-shot notes from PHYSICS_OPERATOR / SESSION_LEADER / CHIEF_OPERATOR
    / DIAGNOSTICS / BEAMS entries, keyed by each entry's own shot number."""
    if relevant_roles is None:
        relevant_roles = ["SESSION_LEADER", "PHYSICS_OPERATOR", "CHIEF_OPERATOR", "DIAGNOSTICS", "BEAMS"]
    log_start = clean_text.find("ELECTRONIC LOGBOOK ENTRIES")
    if log_start == -1:
        return {}
    log_text = clean_text[log_start:]
    matches = list(SHOT_ENTRY_PATTERN.finditer(log_text))
    shot_notes = {}
    for i, m in enumerate(matches):
        shot_number, role = int(m.group(1)), m.group(2)
        if role not in relevant_roles:
            continue
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(log_text)
        note_text = log_text[start:end].strip()
        if note_text:
            shot_notes.setdefault(shot_number, []).append(f"[{role}] {note_text}")
    return {shot: "\n\n".join(notes) for shot, notes in shot_notes.items()}

def get_session_summary_text(clean_text):
    """Narrative summary sections only (not the shot table)."""
    markers = ["SESSION_LEADER SUMMARY", "PHYSICS_OPERATOR SUMMARY", "CHIEF_OPERATOR SUMMARY"]
    starts = [clean_text.find(m) for m in markers if clean_text.find(m) != -1]
    if not starts:
        return ""
    summary_start = min(starts)
    log_start = clean_text.find("ELECTRONIC LOGBOOK ENTRIES")
    summary_end = log_start if log_start != -1 else len(clean_text)
    return clean_text[summary_start:summary_end]

def get_session_shot_list(clean_text):
    table_start = clean_text.find("shots found")
    if table_start == -1:
        return []
    window = clean_text[table_start:table_start + 6000]
    shot_numbers = re.findall(r'\b(1[4-9]\d{4}|2\d{5})\b', window)
    return sorted(set(int(s) for s in shot_numbers))

print("Parsing functions defined")

In [ ]:
def analyze_sentiment(notes):
    """Classical keyword-based sentiment. 'mode'/'clean' excluded - false
    matches on plasma physics vocabulary (O-mode, wall cleaning)."""
    positive_keywords = ["good", "ok", "okay", "runs", "better", "nice", "well",
                          "matched", "repeat", "success", "worked", "stable"]
    negative_keywords = ["trip", "abort", "missed", "issue", "problem", "too far",
                          "culprit", "bad", "failed", "disruption", "unfortunately", ":(", "worse"]
    notes_lower = notes.lower()
    pos = sum(1 for w in positive_keywords if w in notes_lower)
    neg = sum(1 for w in negative_keywords if w in notes_lower)
    sentiment = "positive" if pos > neg else "negative" if neg > pos else "mixed"
    return {"sentiment": sentiment, "positive_count": pos, "negative_count": neg}

print("Sentiment function defined")

In [ ]:
shot_results = []
sessions_with_per_shot_entries = sessions_missing_logbook_section = sessions_missing_html_file = 0
shots_with_own_notes = shots_using_session_fallback = 0

for folder in session_folders:
    html_path = os.path.join(full_logbook_path, folder, "summary.html")
    if not os.path.exists(html_path):
        sessions_missing_html_file += 1
        continue
    with open(html_path, "r") as f:
        clean_text = clean_html(f.read())

    per_shot_notes = parse_session_html_by_shot(clean_text)
    all_session_shots = get_session_shot_list(clean_text)

    if not per_shot_notes:
        sessions_missing_logbook_section += 1
    else:
        sessions_with_per_shot_entries += 1
    if not per_shot_notes and not all_session_shots:
        continue

    session_summary_text = get_session_summary_text(clean_text)
    session_sentiment = analyze_sentiment(session_summary_text) if session_summary_text.strip() else None

    for shot_number, notes in per_shot_notes.items():
        s = analyze_sentiment(notes)
        shot_results.append({"session": folder, "shot_number": shot_number, "sentiment": s["sentiment"],
                              "positive_count": s["positive_count"], "negative_count": s["negative_count"],
                              "source": "per_shot_notes"})
        shots_with_own_notes += 1

    for shot_number in [s for s in all_session_shots if s not in per_shot_notes]:
        if session_sentiment is None:
            continue
        shot_results.append({"session": folder, "shot_number": shot_number, "sentiment": session_sentiment["sentiment"],
                              "positive_count": session_sentiment["positive_count"],
                              "negative_count": session_sentiment["negative_count"], "source": "session_fallback"})
        shots_using_session_fallback += 1

print("Sessions missing summary.html:", sessions_missing_html_file)
print("Sessions with NO logbook section:", sessions_missing_logbook_section)
print("Sessions WITH per-shot entries:", sessions_with_per_shot_entries)
print("Shots from own notes:", shots_with_own_notes, "| via fallback:", shots_using_session_fallback)
print("Total shots scored:", len(shot_results))

output_file = "/path/to/outputs/shot_sentiment.csv"
with open(output_file, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["session", "shot_number", "sentiment", "positive_count", "negative_count", "source"])
    writer.writeheader()
    writer.writerows(shot_results)
print("Saved to:", output_file)